# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Telecom_Customer_Analytics

**Input file:** `01_Telecom_Customer_Analytics.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("01_Telecom_Customer_Analytics.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Dataset profiling

Profile rows, columns, data types and missing values before ETL.

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

## Q2. Active high-usage customers

Find active Premium customers using more than 60 GB and show their bill.

In [ ]:
r=df[(df.plan_type=="Premium")&(df.customer_status=="Active")&(df.data_usage_gb>60)]
print(r[["customer_id","customer_name","city","data_usage_gb","monthly_bill"]].sort_values("data_usage_gb",ascending=False))

## Q3. Selected-city billing

For Delhi, Mumbai, Kolkata and Patna, keep Premium/Standard users with 40–100 GB and calculate total billing.

In [ ]:
r=df[df.city.isin(["Delhi","Mumbai","Kolkata","Patna"])&df.plan_type.isin(["Premium","Standard"])&df.data_usage_gb.between(40,100)]
print(r.groupby("city").monthly_bill.sum().sort_values(ascending=False))

## Q4. Name-based marketing filter

Find active customers whose name contains a and whose bill exceeds 900.

In [ ]:
r=df[df.customer_name.str.contains("a",case=False,na=False)&(df.customer_status=="Active")&(df.monthly_bill>900)]
print(r[["customer_name","city","plan_type","monthly_bill"]])

## Q5. Missing-value audit

Return missing count and percentage for columns containing nulls.

In [ ]:
m=pd.DataFrame({"count":df.isna().sum(),"pct":(df.isna().mean()*100).round(2)})
print(m[m["count"]>0])

## Q6. Plan-level imputation

Fill missing usage and billing using the median of each plan.

In [ ]:
x=df.copy()
x["data_usage_gb"]=x.data_usage_gb.fillna(x.groupby("plan_type").data_usage_gb.transform("median"))
x["monthly_bill"]=x.monthly_bill.fillna(x.groupby("plan_type").monthly_bill.transform("median"))
print(x.isna().sum())

## Q7. Name standardization

Normalize customer names to title case.

In [ ]:
x=df.copy(); x["customer_name"]=x.customer_name.str.strip().str.title(); print(x.customer_name.head(20))

## Q8. Datetime conversion

Convert registration date and report earliest and latest dates.

In [ ]:
x=df.copy(); x["registration_date"]=pd.to_datetime(x.registration_date); print(x.registration_date.min(),x.registration_date.max())

## Q9. City distribution

Count customers per city including missing city values.

In [ ]:
print(df.groupby("city",dropna=False).size().reset_index(name="customers").sort_values("customers",ascending=False))

## Q10. Plan revenue

Calculate count, total revenue and average bill by plan.

In [ ]:
print(df.groupby("plan_type").agg(customers=("customer_id","count"),revenue=("monthly_bill","sum"),avg_bill=("monthly_bill","mean")).sort_values("revenue",ascending=False))

## Q11. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["data_usage_gb"].agg(["count","mean","min","max"]))

## Q12. High-value records

Identify records above the 75th percentile of data_usage_gb.

In [ ]:
t=    customer_id customer_name  ... customer_status registration_date
0         C0001     Sneha Das  ...          Active        2024-12-27
1         C0002   Priya Verma  ...        Inactive        2025-05-05
2         C0003   Neha Sharma  ...          Active        2024-08-21
3         C0004   Vikas Yadav  ...          Active        2024-10-05
4         C0005     Sneha Das  ...          Active        2026-07-30
..          ...           ...  ...             ...               ...
348       C0349    Amit Singh  ...          Active        2025-06-07
349       C0350   Rohit Gupta  ...          Active        2026-03-21
350       C0011    Anjali Roy  ...          Active        2025-02-14
351       C0026   Priya Verma  ...        Inactive        2024-12-20
352       C0071   Rohit Gupta  ...          Active        2025-07-23

[353 rows x 12 columns]
threshold=df["data_usage_gb"].quantile(.75)
print(df[df["data_usage_gb"]>threshold].sort_values("data_usage_gb",ascending=False).head(20))

## Q13. Category aggregation

Calculate record count, total data_usage_gb, and average voice_minutes for each customer_id.

In [ ]:
print(df.groupby("customer_id",dropna=False).agg(count=("data_usage_gb","count"),total=("data_usage_gb","sum"),avg=("voice_minutes","mean")).sort_values("total",ascending=False))

## Q14. Two-dimensional aggregation

Analyze data_usage_gb by customer_id and customer_name; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["customer_id","customer_name"],dropna=False).agg(count=("data_usage_gb","count"),total=("data_usage_gb","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q15. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["data_usage_gb"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q16. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="customer_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q17. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("customer_id"); print("Before:",len(df),"After:",len(x))

## Q18. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q19. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["customer_id"]=x["customer_id"].fillna("Unknown"); print(x["customer_id"].value_counts(dropna=False))

## Q20. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["data_usage_gb"]=x["data_usage_gb"].fillna(x["data_usage_gb"].median()); print(x["data_usage_gb"].isna().sum())

## Q21. String search

Find records where customer_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["customer_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q22. Sorted analysis

Sort the dataset by data_usage_gb descending and voice_minutes ascending.

In [ ]:
print(df.sort_values(["data_usage_gb","voice_minutes"],ascending=[False,True]).head(25))

## Q23. Conditional segmentation

Create a new segment based on data_usage_gb using its median as the threshold.

In [ ]:
x=df.copy(); med=x["data_usage_gb"].median(); x["segment"]=np.where(x["data_usage_gb"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q24. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("customer_id")["data_usage_gb"].transform("mean"); print(x[["customer_id","data_usage_gb","group_avg"]].head(25))

## Q25. Group share

Calculate each customer_id's share of total data_usage_gb.

In [ ]:
g=df.groupby("customer_id",dropna=False)["data_usage_gb"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q26. Pivot report

Create a pivot table showing average data_usage_gb by customer_id and customer_name.

In [ ]:
print(pd.pivot_table(df,index="customer_id",columns="customer_name",values="data_usage_gb",aggfunc="mean"))

## Q27. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("customer_id")[["data_usage_gb","voice_minutes"]].mean().reset_index(); print(w.melt(id_vars="customer_id",var_name="metric",value_name="value"))

## Q28. Validation rule

Count records where data_usage_gb is negative or missing.

In [ ]:
print("Invalid:",df["data_usage_gb"].isna().sum()+(df["data_usage_gb"]<0).sum())

## Q29. Outlier detection

Use IQR to identify high outliers in data_usage_gb.

In [ ]:
q1=df["data_usage_gb"].quantile(.25); q3=df["data_usage_gb"].quantile(.75); iqr=q3-q1; print(df[df["data_usage_gb"]>q3+1.5*iqr][["data_usage_gb"]].head(30))

## Q30. Top group

Find the category in customer_id with the highest total data_usage_gb.

In [ ]:
print(df.groupby("customer_id")["data_usage_gb"].sum().nlargest(1))

## End of Project

The notebook is ready to execute with the supplied input CSV.